In [1]:
import os
import shutil
import numpy as np

In [2]:
import numpy as np

def calculate_normal_metrics(normals_pred, normals_gt, depths_valid_mask=None):
    """
    표면 normal에 대한 메트릭을 계산합니다. (두 입력이 동일 좌표계라고 가정)

    Args:
        normals_pred (np.array): 예측된 normal, shape (H, W, 3)
        normals_gt (np.array): Ground Truth normal, shape (H, W, 3)

    Returns:
        dict: MAE 및 threshold accuracy를 포함하는 메트릭 딕셔너리
    """
    assert normals_pred.shape == normals_gt.shape
    
    # 1. 유효한 GT normal에 대한 마스크 생성 (배경 등 제외)
    gt_norm = np.linalg.norm(normals_gt, axis=2)
    mask = gt_norm > 1e-6
    
    if depths_valid_mask is not None:
        mask = mask & depths_valid_mask

    # 마스크를 적용하여 유효한 픽셀만 추출
    normals_pred_flat = normals_pred[mask]
    normals_gt_flat = normals_gt[mask]
    
    # 2. Normal 벡터 정규화 (단위 벡터로 만들기)
    norm_pred = np.linalg.norm(normals_pred_flat, axis=1, keepdims=True)
    norm_gt = np.linalg.norm(normals_gt_flat, axis=1, keepdims=True)
    
    # norm이 0인 경우를 방지
    norm_pred[norm_pred < 1e-6] = 1e-6
    norm_gt[norm_gt < 1e-6] = 1e-6

    normals_pred_unit = normals_pred_flat / norm_pred
    normals_gt_unit = normals_gt_flat / norm_gt

    # 3. 각도 오차 계산
    dot_product = np.sum(normals_pred_unit * normals_gt_unit, axis=1)
    dot_product = np.clip(dot_product, -1.0, 1.0)
    
    angular_error_deg = np.degrees(np.arccos(dot_product))

    # 4. 메트릭 계산
    mae = np.mean(angular_error_deg)
    
    total_valid_pixels = len(angular_error_deg)
    acc_11_25 = np.sum(angular_error_deg < 11.25) / total_valid_pixels
    acc_22_5 = np.sum(angular_error_deg < 22.5) / total_valid_pixels
    acc_30 = np.sum(angular_error_deg < 30.0) / total_valid_pixels
    
    metrics = {
        'mae': mae,
        'acc_11.25': acc_11_25 * 100,
        'acc_22.5': acc_22_5 * 100,
        'acc_30': acc_30 * 100,
    }
    
    return metrics

def compute_errors(gt, pred):
    """Computation of error metrics between predicted and ground truth depths
    """
    thresh = np.maximum((gt / pred), (pred / gt))
    a1 = (thresh < 1.25     ).mean()
    a2 = (thresh < 1.25 ** 2).mean()
    a3 = (thresh < 1.25 ** 3).mean()

    rmse = (gt - pred) ** 2
    rmse = np.sqrt(rmse.mean())

    rmse_log = (np.log(gt) - np.log(pred)) ** 2
    rmse_log = np.sqrt(rmse_log.mean())

    abs_rel = np.mean(np.abs(gt - pred) / gt)

    sq_rel = np.mean(((gt - pred) ** 2) / gt)

    return abs_rel, sq_rel, rmse, rmse_log, a1, a2, a3

import numpy as np

def calculate_depth_fscore(depth_pred, depth_gt, delta_threshold=1.25):
    """
    Depth map을 기반으로 F-score를 계산합니다.
    "성공"의 기준을 상대적인 오차(delta threshold)로 변경하여 더 안정적인 측정이 가능합니다.

    Args:
        depth_pred (np.array): 예측 깊이 맵
        depth_gt (np.array): 실제(GT) 깊이 맵
        delta_threshold (float): 성공/실패를 가르는 상대 오차 임계값 (예: 1.25)

    Returns:
        dict: 정밀도, 재현율, F-score가 담긴 딕셔너리
    """
    # 0으로 나누는 것을 방지하기 위해 아주 작은 값을 추가
    gt_safe = np.maximum(depth_gt, 1e-6)
    pred_safe = np.maximum(depth_pred, 1e-6)
    
    # 유효한 픽셀 마스크
    gt_valid_mask = depth_gt > 1e-6
    pred_valid_mask = depth_pred > 1e-6
    
    # 성공(Correct) 픽셀 마스크: 상대 오차(delta) 기준으로 변경
    thresh = np.maximum((gt_safe / pred_safe), (pred_safe / gt_safe))
    correct_mask = thresh < delta_threshold

    # TP: GT가 유효하고, 예측도 성공한 경우
    tp_mask = np.logical_and(gt_valid_mask, correct_mask)
    tp = tp_mask.sum()

    # FN: GT는 유효하지만, 예측은 실패한 경우
    fn_mask = np.logical_and(gt_valid_mask, ~correct_mask)
    fn = fn_mask.sum()
    
    # FP: GT는 유효하지 않지만, 예측은 유효하다고 한 경우
    fp_mask = np.logical_and(~gt_valid_mask, pred_valid_mask)
    fp = fp_mask.sum()

    # 메트릭 계산
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    fscore = 0
    if precision + recall > 0:
        fscore = 2 * (precision * recall) / (precision + recall)
        
    metrics = {
        'precision': precision * 100,
        'recall': recall * 100,
        'f1_score': fscore * 100
    }
    return metrics


import imageio.v2 as imageio
import matplotlib.pyplot as plt

def visualize_depth(depth, percentile=99):
    """1채널 Depth 맵을 컬러 이미지로 변환합니다."""
    valid_mask = depth > 0
    if not valid_mask.any():
        return np.zeros((depth.shape[0], depth.shape[1], 3), dtype=np.uint8)
    vmax = np.percentile(depth[valid_mask], percentile)
    normalized_depth = np.clip(depth / vmax, 0, 1)
    colored_depth = (plt.cm.viridis(normalized_depth)[:, :, :3] * 255).astype(np.uint8)
    colored_depth[~valid_mask] = 0
    return colored_depth

def visualize_normals(normals):
    """3채널 Normal 맵(-1~1)을 컬러 이미지(0~255)로 변환합니다."""
    norm_img = ((normals * 0.5 + 0.5) * 255).astype(np.uint8)
    valid_mask = np.linalg.norm(normals, axis=2) > 1e-6
    norm_img[~valid_mask] = 0
    return norm_img

def range_to_camZ(range_map, fx, fy, cx, cy):
    H, W = range_map.shape
    u = np.arange(W)[None, :].repeat(H, axis=0)
    v = np.arange(H)[:, None].repeat(W, axis=1)
    dx = (u - cx) / fx
    dy = (v - cy) / fy
    dz = 1.0 / np.sqrt(dx**2 + dy**2 + 1.0)  # cos(theta)
    return range_map * dz

def load_colmap_intrinsics(cameras_txt_path):
    # cameras.txt 한 줄: id MODEL W H fx fy cx cy
    with open(cameras_txt_path, "r") as f:
        line = [ln for ln in f if ln.strip() and not ln.startswith("#")][0]
    _, _, W, H, fx, fy, cx, cy = line.split()
    W, H = int(float(W)), int(float(H))
    fx, fy, cx, cy = map(float, (fx, fy, cx, cy))
    K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]], dtype=np.float64)
    return K, W, H

def scale_intrinsics(K, W, H, scale):
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    fx, fy = fx*scale, fy*scale
    cx, cy = cx*scale, cy*scale
    W, H = int(W*scale), int(H*scale)
    K_new = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]], dtype=np.float64)
    return K_new, W, H


In [3]:
def convert_diffren_to_stable(normals_d):
    ss = np.empty_like(normals_d)
    ss[..., 0] =  normals_d[..., 0]  
    ss[..., 1] = -normals_d[..., 1]  
    ss[..., 2] = -normals_d[..., 2] 
    # (필요하면 다시 정규화)
    norm = np.linalg.norm(ss, axis=2, keepdims=True)
    norm[norm < 1e-6] = 1.0
    return ss / norm

import torch
def normalize(x, eps):
    # channel last: normalization
    return x / (torch.norm(x, dim=-1, keepdim=True) + eps)

def normalize_np(x, eps=1e-6):
    # channel last: normalization
    ss = x
    norm = np.linalg.norm(ss, axis=2, keepdims=True)
    
    
import numpy as np

def unit_dz_map(W, H, fx, fy, cx, cy):
    """OpenCV(+Z forward) 핀홀에서 단위 시선벡터의 z성분 = cosθ = 1/||[(x,y,1)]||"""
    u = np.arange(W, dtype=np.float32)
    v = np.arange(H, dtype=np.float32)
    uu, vv = np.meshgrid(u, v)
    x = (uu + 0.5 - cx) / fx
    y = (vv + 0.5 - cy) / fy
    denom = np.sqrt(x * x + y * y + 1.0)
    return 1.0 / denom  # = cosθ

def align_pred_to_camZ(pred_depth, K, W, H, gt_camZ, valid_mask=None):
    """
    예측 depth가 어떤 convention인지 모를 때,
    몇 가지 후보(있는 그대로 / 거리 / 역심도 / 역거리)를 시험해 보고
    (a,b) 선형 스케일 보정까지 포함해 GT(camera-Z)와 가장 잘 맞는 것을 선택.
    반환: (best_pred_camZ, best_mode, mae)
    """
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    dz = unit_dz_map(W, H, fx, fy, cx, cy)
    eps = 1e-8

    candidates = {
        "as_is": pred_depth,                      # 이미 camera-Z일 수도 있음
        "ray_distance": pred_depth * dz,         # 예측이 '거리'라면 camera-Z = distance * cosθ
        "inverse_depth": 1.0 / np.maximum(pred_depth, eps),           # 예측이 1/Z
        "inverse_ray_distance": (1.0 / np.maximum(pred_depth, eps)) * dz,  # 예측이 1/distance
    }

    vm = np.isfinite(gt_camZ)
    if valid_mask is not None: vm &= valid_mask
    best = (None, None, np.inf)

    for name, cand in candidates.items():
        if cand.shape != gt_camZ.shape:
            cand = cand.reshape(gt_camZ.shape)  # 당신 코드 스타일 유지
        m = vm & np.isfinite(cand)
        if not np.any(m): 
            continue
        # gt ≈ a * cand + b  (전역 스케일/오프셋 불일치도 흡수)
        x = cand[m].reshape(-1)
        y = gt_camZ[m].reshape(-1)
        A = np.vstack([x, np.ones_like(x)]).T
        a, b = np.linalg.lstsq(A, y, rcond=None)[0]
        est = a * cand + b
        mae = np.nanmean(np.abs(est[m] - y))
        if mae < best[2]:
            best = (est, name, mae)

    return best  # (best_pred_camZ, best_mode, best_mae)

In [4]:
# pred path
# pred_path = "data/result/envgs/manual_synthetic_GT_cam_points/0902-GT-cam-points-final"
pred_path = "data/result/envgs/manual_synthetic_GT_cam/0925-manual-synthetic"
# gt path
gt_path = "/home/youngju/ssd/datasets/nerfstudio/GT_cam"
# os.listdir(os.path.join(gt_path, "scene_5", "depths_gt"))

# Normal Metrics

In [ ]:
import numpy as np
import os
from collections import defaultdict
from PIL import Image

for scene in sorted(os.listdir(pred_path)):
    
    print(f"Processing Scene: {scene}")
    normals_pred_files = sorted(os.listdir(os.path.join(pred_path, scene, "geo_metric", "normals_pred")))
    depths_pred_files = sorted(os.listdir(os.path.join(pred_path, scene, "geo_metric", "depths_pred")))

    scene_metrics_normal = defaultdict(list)
    scene_metrics_depth = defaultdict(list)
    scene_metrics_depth_fscore = defaultdict(list)

    # normals_gt_files = sorted(os.listdir(os.path.join(gt_path, scene, "normals_gt")))
    
    for depth_pred_file in depths_pred_files:
        frame_idx = int(depth_pred_file.split(".")[0].split("_")[-1])
        # print(f"  Frame Index: {frame_idx}")
        gt_frame_idx = frame_idx #!* 8 # 0, 8, 16, ... -> 0, 1, 2, ...
        
        # --- 1. Depth 데이터 로드 및 전처리 ---
        depth_pred_path = os.path.join(pred_path, scene, "geo_metric", "depths_pred", depth_pred_file)
        depth_gt_path = os.path.join(gt_path, scene, "depths_gt", f"val_depthCamZ_{gt_frame_idx:04d}.npy")
        assert os.path.exists(depth_gt_path), f"GT depth file not found: {depth_gt_path}"
        
        depths_pred_raw = np.load(depth_pred_path) # (b, hw, 1) or (b, h, w, 1)
        depths_gt_raw = np.load(depth_gt_path) # (h, w) or (h, w, 1)

        # GT를 기준으로 shape 통일
        depths_gt = depths_gt_raw.squeeze()[::2, ::2] # 다운샘플링
        depths_pred = depths_pred_raw.reshape(depths_gt.shape)
        
        assert depths_pred.shape == depths_gt.shape, f"Depth shape mismatch: {depths_pred.shape} vs {depths_gt.shape}"
        
        # --- 2. Normal 데이터 로드 및 전처리 (Depth 마스크 사용) ---
        normal_pred_path = os.path.join(pred_path, scene, "geo_metric", "normals_pred", f"{frame_idx:04d}.npy")
        normal_gt_path = os.path.join(gt_path, scene, "normals_gt", f"val_normal_{frame_idx:04d}.npy")
        
        # print(normal_pred_path, normal_gt_path)
        assert os.path.exists(normal_pred_path) and os.path.exists(normal_gt_path)
        
        normals_pred_raw = np.load(normal_pred_path)[0]
        normals_gt_raw = np.load(normal_gt_path)
        
        normals_gt = normals_gt_raw[::2, ::2]
        normals_pred = normals_pred_raw.reshape(normals_gt.shape)

        # --- 3. 메트릭 계산 및 누적 ---
        depth_valid_mask = depths_gt > 1e-6
        # visualize both
        vis_pred = visualize_normals(normals_pred)
        vis_gt = visualize_normals(normals_gt)
        # acc_11.25 에러 픽셀 시각화 (빨간색으로 추가)
        norm_metrics = calculate_normal_metrics(normals_pred, normals_gt, depth_valid_mask)
        gt_norm = np.linalg.norm(normals_gt, axis=2)
        mask = (gt_norm > 1e-6) & depth_valid_mask
        dot = np.sum((normals_pred * normals_gt), axis=2) / (np.linalg.norm(normals_pred, axis=2) * gt_norm + 1e-6)
        angular_error = np.degrees(np.arccos(np.clip(dot, -1, 1)))
        error_threshold = 11.5
        error_vis = np.zeros_like(vis_gt); error_vis[mask & (angular_error >= error_threshold)] = [255, 0, 0]; error_vis[mask & (angular_error < error_threshold)] = [0, 255, 0]
        vis = np.concatenate([vis_gt, vis_pred, error_vis], axis=1)
        plt.imshow(vis); plt.show()
        break
        
        # --- 3. 메트릭 계산 및 누적 ---
        depth_valid_mask = depths_gt > 1e-6
    
        # Normal 메트릭 계산 (유효한 깊이 영역만 사용)
        norm_metrics = calculate_normal_metrics(normals_pred, normals_gt, depths_valid_mask=depth_valid_mask)
        for key, value in norm_metrics.items():
            scene_metrics_normal[key].append(value)
            
        # print(f"  Frame {frame_idx}: MAE={norm_metrics['mae']:.2f}°, Acc<11.25°={norm_metrics['acc_11.25']:.2f}%, Acc<22.5°={norm_metrics['acc_22.5']:.2f}%, Acc<30°={norm_metrics['acc_30']:.2f}%")

        
        # Depth 메트릭 계산
        depth_metrics = compute_errors(depths_pred, depths_gt)
        abs_rel, sq_rel, rmse, rmse_log, a1, a2, a3 = depth_metrics
        depth_metrics = {
            'abs_rel': abs_rel,
            'sq_rel': sq_rel,
            'rmse': rmse,
            'rmse_log': rmse_log,
            'acc_delta_1.25': a1 * 100,
            'acc_delta_1.25_2': a2 * 100,
            'acc_delta_1.25_3': a3 * 100,
        }
        
        for key, value in depth_metrics.items():
            scene_metrics_depth[key].append(value)
            
        # print(f"  Frame {frame_idx}: AbsRel={depth_metrics['abs_rel']:.3f}, RMSE={depth_metrics['rmse']:.3f}")

    # --- 4. 씬 전체 평균 메트릭 계산 및 출력 ---
    avg_norm_metrics = {key: np.mean(values) for key, values in scene_metrics_normal.items()}
    avg_depth_metrics = {key: np.mean(values) for key, values in scene_metrics_depth.items()}
    avg_depth_metrics_fscore = {key: np.mean(values) for key, values in scene_metrics_depth.items() if key in ['precision', 'recall', 'f1_score']}
    
    print("\n" + "="*60)
    print(f"Quantitative Evaluation for Scene: {scene}")
    print("-" * 60)
    print("Normals:")
    print(f"  Mean Angular Error (MAE↓): {avg_norm_metrics.get('mae', 0):.2f}°")
    print(f"  Accuracy < 11.25° (↑):     {avg_norm_metrics.get('acc_11.25', 0):.2f}%")
    print(f"  Accuracy < 22.5° (↑):      {avg_norm_metrics.get('acc_22.5', 0):.2f}%")
    print(f"  Accuracy < 30° (↑):        {avg_norm_metrics.get('acc_30', 0):.2f}%")
    
    
  
    print("="*60 + "\n")
    
        
    

        

Processing Scene: scene_4

Quantitative Evaluation for Scene: scene_4
------------------------------------------------------------
Normals:
  Mean Angular Error (MAE↓): 17.89°
  Accuracy < 11.25° (↑):     57.23%
  Accuracy < 22.5° (↑):      74.57%
  Accuracy < 30° (↑):        80.64%

Processing Scene: scene_5

Quantitative Evaluation for Scene: scene_5
------------------------------------------------------------
Normals:
  Mean Angular Error (MAE↓): 22.49°
  Accuracy < 11.25° (↑):     39.99%
  Accuracy < 22.5° (↑):      60.33%
  Accuracy < 30° (↑):        70.63%

Processing Scene: scene_6

Quantitative Evaluation for Scene: scene_6
------------------------------------------------------------
Normals:
  Mean Angular Error (MAE↓): 20.31°
  Accuracy < 11.25° (↑):     49.71%
  Accuracy < 22.5° (↑):      65.98%
  Accuracy < 30° (↑):        73.20%

Processing Scene: scene_7

Quantitative Evaluation for Scene: scene_7
------------------------------------------------------------
Normals:
  Mea

In [7]:
import numpy as np
import os
from collections import defaultdict
import matplotlib

# =========================
# Config
# =========================
pred_root = pred_path  # 기존 변수 사용
gt_root   = gt_path    # 기존 변수 사용

colmap_scale_factor = 1.0
correction_factor = 1.0 / colmap_scale_factor   # 예: 2.0 이면 pred를 1/2 배로
to_meter = 1/1.0   # pred가 mm면 /1000, 이미 m면 1.0
DEBUG_VIS = False     # True면 중간 depth 시각화(당신 함수 visualize_depth 사용) 

# =========================
# Utils
# =========================
def down2_avg(img):
    """H×W 또는 H×W×C 를 2x2 영역평균으로 다운샘플"""
    img = np.asarray(img, dtype=np.float32)
    if img.ndim == 2:
        return 0.25 * (img[0::2,0::2] + img[1::2,0::2] + img[0::2,1::2] + img[1::2,1::2])
    elif img.ndim == 3:
        return 0.25 * (img[0::2,0::2,:] + img[1::2,0::2,:] + img[0::2,1::2,:] + img[1::2,1::2,:])
    else:
        raise ValueError(f"Unexpected ndim={img.ndim}")

def valid_mask(pred, gt, min_gt=1e-6, max_gt=20.0):
    vm = np.isfinite(gt) & np.isfinite(pred) & (gt > min_gt) & (pred > 0) & (gt < max_gt) & (pred < max_gt)
    return vm

def median_scale_align(pred, gt, vm):
    """frame별 median-scaling s = median(gt)/median(pred)"""
    med_p = np.median(pred[vm])
    med_g = np.median(gt[vm])
    if med_p <= 0:
        return 1.0
    return med_g / med_p

def compute_depth_metrics(gt, pr):
    """표준 metric 세트 (AbsRel, SqRel, RMSE, RMSE_log, δ1/δ2/δ3)"""
    eps = 1e-12
    diff = pr - gt
    abs_rel = np.mean(np.abs(diff) / (gt + eps))
    sq_rel  = np.mean((diff ** 2) / (gt + eps))
    rmse    = np.sqrt(np.mean(diff ** 2))
    rmse_log = np.sqrt(np.mean((np.log(pr + eps) - np.log(gt + eps)) ** 2))

    ratio = np.maximum(pr / (gt + eps), gt / (pr + eps))
    a1 = np.mean(ratio < 1.25)
    a2 = np.mean(ratio < 1.25 ** 2)
    a3 = np.mean(ratio < 1.25 ** 3)

    return {
        'abs_rel': abs_rel,
        'sq_rel': sq_rel,
        'rmse': rmse,
        'rmse_log': rmse_log,
        'acc_delta_1.25': a1 * 100,
        'acc_delta_1.25_2': a2 * 100,
        'acc_delta_1.25_3': a3 * 100,
    }
    
# =========================
# Main
# =========================

for scene in sorted(os.listdir(pred_path)):

    if scene == "scene_3":
        continue
    
    print(f"Processing Scene: {scene}")
    normals_pred_files = sorted(os.listdir(os.path.join(pred_path, scene, "geo_metric", "normals_pred")))
    depths_pred_files = sorted(os.listdir(os.path.join(pred_path, scene, "geo_metric", "depths_pred")))

    scene_metrics_normal = defaultdict(list)
    # scene-wise aggregator
    scene_metrics_depth = defaultdict(list)
    mean_med_ratio = []
    scene_metrics_depth_fscore = defaultdict(list)

    # normals_gt_files = sorted(os.listdir(os.path.join(gt_path, scene, "normals_gt")))

    for depth_pred_file in depths_pred_files:
        frame_idx = int(depth_pred_file.split(".")[0].split("_")[-1])
        # print(f"  Frame Index: {frame_idx}")
        gt_frame_idx = frame_idx #!* 8 # 0, 8, 16, ... -> 0, 1, 2, ...
        
        # --- 1. Depth 데이터 로드 및 전처리 ---
        depth_pred_path = os.path.join(pred_path, scene, "geo_metric", "depths_pred", depth_pred_file)
        depth_gt_path = os.path.join(gt_path, scene, "depths_gt", f"val_depthCamZ_{gt_frame_idx:04d}.npy")
        assert os.path.exists(depth_gt_path), f"GT depth file not found: {depth_gt_path}"
        
        depths_pred_raw = np.load(depth_pred_path) # (b, hw, 1) or (b, h, w, 1)
        depths_gt_raw = np.load(depth_gt_path) # (h, w) or (h, w, 1)

        # GT를 기준으로 shape 통일
        

        # print("scale of depths_pred_raw:", np.min(depths_pred_raw), np.max(depths_pred_raw), np.mean(depths_pred_raw), "GT:", np.min(depths_gt_raw), np.max(depths_gt_raw), np.mean(depths_gt_raw))

        # --- preprocess/reshape/unit ---
        depths_gt = down2_avg(depths_gt_raw.squeeze().astype(np.float32))  # GT 단위가 m이라고 가정
        depths_pred = np.asarray(depths_pred_raw, dtype=np.float32).reshape(depths_gt.shape)
        depths_pred = depths_pred / correction_factor
        depths_pred = depths_pred * to_meter  # (pred가 mm면 /1000, 이미 m이면 1.0로 두세요)

        assert depths_pred.shape == depths_gt.shape, f"Depth shape mismatch: {depths_pred.shape} vs {depths_gt.shape}"
        
        
        # sanity log
        # print(f"  Frame {frame_idx:4d}: pred=({depths_pred.min():.3f},{depths_pred.max():.3f})  "
            #   f"gt=({depths_gt.min():.3f},{depths_gt.max():.3f})")

        # --- valid mask ---
        vm = valid_mask(depths_pred, depths_gt)
        if not np.any(vm):
            print(f"    [WARN] no valid pixels")
            continue

        # --- frame-level median scaling ---
        s = median_scale_align(depths_pred, depths_gt, vm)
        depths_pred_aligned = depths_pred * s
        
        
        # optional debug vis
        if DEBUG_VIS:
            vis_pred = visualize_depth(depths_pred_aligned)  # 사용자 정의 함수
            vis_gt   = visualize_depth(depths_gt)    # 사용자 정의 함수
            # 차이 시각화 (절대값 차이, 컬러맵)
            diff = np.abs(depths_pred_aligned - depths_gt)
            import matplotlib
            import matplotlib.pyplot as plt
            diff_norm = (diff - diff.min()) / (diff.max() - diff.min() + 1e-8)
            diff_color = matplotlib.cm.jet(diff_norm)[..., :3]  # RGB, shape (H,W,3), float
            diff_color = (diff_color * 255).astype(np.uint8)
            vis = np.concatenate([vis_gt, vis_pred, diff_color], axis=1)
            plt.imshow(vis)
            plt.title(f"{scene} | frame {frame_idx} (GT | Pred | AbsDiff)")
            plt.axis('off')
            plt.show()
            
            break
            

        # 중앙경향 체크(선택)
        mean_med_ratio.append(float(np.median(depths_pred_aligned[vm]) / (np.median(depths_gt[vm]) + 1e-12)))

        # --- metrics ---
        m = compute_depth_metrics(depths_gt[vm], depths_pred_aligned[vm])
        for k, v in m.items():
            scene_metrics_depth[k].append(v)

    # =========================
    # Scene-level summary
    # =========================
    if len(scene_metrics_depth) == 0 or len(next(iter(scene_metrics_depth.values()))) == 0:
        print(f"[INFO] No frames evaluated in scene {scene}")
        continue

    avg = {k: float(np.mean(v)) for k, v in scene_metrics_depth.items()}
    print("\n" + "="*64)
    print(f"Quantitative Evaluation (Median-Scaled) — Scene: {scene}")
    print("-"*64)
    print(f"AbsRel↓      : {avg['abs_rel']:.3f}")
    print(f"SqRel↓       : {avg['sq_rel']:.3f}")
    print(f"RMSE↓        : {avg['rmse']:.3f}")
    print(f"RMSE (log)↓  : {avg['rmse_log']:.3f}")
    print(f"δ < 1.25  ↑  : {avg['acc_delta_1.25']:.2f}%")
    print(f"δ < 1.25² ↑  : {avg['acc_delta_1.25_2']:.2f}%")
    print(f"δ < 1.25³ ↑  : {avg['acc_delta_1.25_3']:.2f}%")
    if len(mean_med_ratio):
        print(f"median(pred_aligned)/median(gt) ≈ {np.mean(mean_med_ratio):.3f} (≈1이 이상적)")
    print("="*64 + "\n")


Processing Scene: scene_4

Quantitative Evaluation (Median-Scaled) — Scene: scene_4
----------------------------------------------------------------
AbsRel↓      : 0.072
SqRel↓       : 0.033
RMSE↓        : 0.304
RMSE (log)↓  : 0.115
δ < 1.25  ↑  : 96.79%
δ < 1.25² ↑  : 99.04%
δ < 1.25³ ↑  : 99.56%
median(pred_aligned)/median(gt) ≈ 1.000 (≈1이 이상적)

Processing Scene: scene_5

Quantitative Evaluation (Median-Scaled) — Scene: scene_5
----------------------------------------------------------------
AbsRel↓      : 0.070
SqRel↓       : 0.024
RMSE↓        : 0.268
RMSE (log)↓  : 0.100
δ < 1.25  ↑  : 97.35%
δ < 1.25² ↑  : 99.60%
δ < 1.25³ ↑  : 99.81%
median(pred_aligned)/median(gt) ≈ 1.000 (≈1이 이상적)

Processing Scene: scene_6

Quantitative Evaluation (Median-Scaled) — Scene: scene_6
----------------------------------------------------------------
AbsRel↓      : 0.063
SqRel↓       : 0.019
RMSE↓        : 0.229
RMSE (log)↓  : 0.085
δ < 1.25  ↑  : 98.42%
δ < 1.25² ↑  : 99.69%
δ < 1.25³ ↑  : 99.86%
m